---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("Prob_Q")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / Prob_Q.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

# 商業類術科精選｜配合 Lesson 6

對應 **Lesson6**：`基礎演算法(1).ipynb`（排序與反复合併／貪婪策略）。

題組來自術科精選（112–113，Lesson3 範圍），依本週教材主題分配，**不重複**。

**本檔收錄：**
- **112 學年度 Problem Q**


## Problem Q 數字全部相加成本

### 題目（PDF 擷取）

```text
Problem Description
給你一個數列，將數字全部相加，相加的過程需要成本，例如1 + 3 = 4，則成本為4，請
善用𝑎+ 𝑏= 𝑐, cost += 𝑐，其中𝑎與𝑏是數列裡面的數字，並將𝑐再放入數列，請求出最小
的cost，求將數字全部相加後的成本為何？
假如你想要加1, 2 和3，那麼有以下幾種方法：
1 + 2 = 3, cost = 3, 3 + 3 = 6, cost = 6, Total cost = 9
1 + 3 = 4, cost = 4, 2 + 4 = 6, cost = 6, Total cost = 10
2 + 3 = 5, cost = 5, 1 + 5 = 6, cost = 6, Total cost = 11
你已經瞭解你的任務，就是把𝑁個數加起來使得付出的代價最少。
Input Format
輸入含有多組測試資料。每組輸入有二列，每組測試資料第一列有一個正整數𝑁(2 ≤𝑁≤
104)，第二列有𝑁個由空白隔開的正整數，（均小於106）。
當𝑁= 0 時代表輸入結束。
Output Format
對每一組測試資料輸出一列，把𝑁個數加起來使得付出的代價最少是多少？
Sample Input 1
3
1 2 3
4
1 2 3 4
0
Sample Output 1
9
19
Hint
Huffman編碼。每次選擇最小的兩者𝑎, 𝑏相加，請善用𝑎+ 𝑏= 𝑐, cost += 𝑐，其中𝑎與𝑏是數
列裡面的數字，並將𝑐再放入數列，即可花費最少的成本。
```

### 參考解答


In [ ]:
#start-Prob_Q


In [ ]:
make_submit_button("Prob_Q")
